# Fresh Start NBA — Notebook 5: TensorFlow Experimental

Two deep learning models, compared side-by-side against your XGBoost baseline:

**Model A — Tabular MLP:** Same 178 features as XGBoost, fed into a dense neural network.
Simplest test — does a neural net beat a gradient boosted tree on tabular sports data?

**Model B — LSTM Sequence Model:** Each player's last 10 games are used as a time-series
sequence. The LSTM learns momentum and trend patterns that rolling averages approximate
but never fully capture.

**Enable GPU:** Runtime → Change runtime type → T4 GPU (free tier)

**Prerequisites:** Run notebook 3 first to generate `outputs/nba_features.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

BASE_DIR   = Path('/content/drive/MyDrive/Fresh_Start_NBA_Colab')
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
OUT_DIR    = BASE_DIR / 'outputs'

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110

print(f'TensorFlow version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}  ({[g.name for g in gpus]})')

## Load Data

In [ ]:
feat_path = OUT_DIR / 'nba_features.csv'
if not feat_path.exists():
    raise FileNotFoundError('Run notebook 3 first to generate nba_features.csv')

df = pd.read_csv(feat_path, low_memory=False)
df['game_date'] = pd.to_datetime(df['game_date'])
df = df.sort_values('game_date').reset_index(drop=True)
print(f'Loaded: {df.shape}')

with open(BASE_DIR / 'feature_cols_advanced.json') as f:
    feat_meta = json.load(f)
ALL_FEAT_COLS = feat_meta['feature_columns']
FEATURE_COLS = [c for c in ALL_FEAT_COLS if c in df.columns]
print(f'Features: {len(FEATURE_COLS)}/{len(ALL_FEAT_COLS)}')

# Stat to model — change this to switch targets
TARGET_STAT = 'pts'   # options: pts, trb, ast, pa, tov
MIN_GAMES   = 10
HOLDOUT_DAYS = 30

df_clean = df.dropna(subset=[TARGET_STAT] + FEATURE_COLS[:10]).copy()
if 'games_played' in df_clean.columns:
    df_clean = df_clean[df_clean['games_played'] >= MIN_GAMES]
print(f'Clean rows for {TARGET_STAT}: {len(df_clean):,}')

## Train / Test Split & Scaling

In [ ]:
cutoff = df_clean['game_date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train_df = df_clean[df_clean['game_date'] <= cutoff].copy()
test_df  = df_clean[df_clean['game_date'] >  cutoff].copy()

X_train = train_df[FEATURE_COLS].fillna(0).values.astype(np.float32)
y_train = train_df[TARGET_STAT].values.astype(np.float32)
X_test  = test_df[FEATURE_COLS].fillna(0).values.astype(np.float32)
y_test  = test_df[TARGET_STAT].values.astype(np.float32)

# Neural nets require feature scaling — XGBoost does not
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')
print(f'Target range: {y_train.min():.1f} – {y_train.max():.1f}  mean={y_train.mean():.1f}')

---
# Model A — Tabular MLP

Architecture:
```
Input (178 features)
  → Dense(256, relu) → BatchNorm → Dropout(0.3)
  → Dense(128, relu) → BatchNorm → Dropout(0.3)
  → Dense(64, relu)
  → Dense(1)  [regression]
```

In [ ]:
def build_mlp(input_dim, dropout_rate=0.3, l2_reg=1e-4):
    reg = regularizers.l2(l2_reg)
    inp = keras.Input(shape=(input_dim,), name='features')

    x = layers.Dense(256, activation='relu', kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(128, activation='relu', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)

    out = layers.Dense(1, name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='NBA_MLP')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='mae'
    )
    return model


mlp = build_mlp(X_train_sc.shape[1])
mlp.summary()

In [ ]:
mlp_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1)
]

mlp_history = mlp.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=512,
    callbacks=mlp_callbacks,
    verbose=1
)

# Evaluate
mlp_preds = mlp.predict(X_test_sc, verbose=0).flatten()
mlp_mae = mean_absolute_error(y_test, mlp_preds)

line_col = f'{TARGET_STAT}_l10'
if line_col in test_df.columns:
    proxy_lines = test_df[line_col].fillna(test_df[TARGET_STAT].mean()).values
    mlp_acc = ((mlp_preds > proxy_lines) == (y_test > proxy_lines)).mean()
else:
    mlp_acc = None

print(f'\nMLP  MAE: {mlp_mae:.3f}')
if mlp_acc: print(f'MLP  OVER/UNDER Accuracy: {mlp_acc*100:.1f}%')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(mlp_history.history['loss'],     label='Train loss', linewidth=2)
axes[0].plot(mlp_history.history['val_loss'], label='Val loss',   linewidth=2)
axes[0].set_title('MLP Training Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE')
axes[0].legend()

axes[1].scatter(y_test[:500], mlp_preds[:500], alpha=0.4, s=20, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_title(f'MLP Prediction vs Actual ({TARGET_STAT.upper()})', fontweight='bold')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / f'mlp_{TARGET_STAT}_results.png', bbox_inches='tight')
plt.show()

---
# Model B — LSTM Sequence Model

Instead of a single row of features, each player's **last 10 games** are fed in as
a sequence. The LSTM learns temporal patterns.

Architecture:
```
Input (10 timesteps × N per-game features)
  → LSTM(128) → Dropout(0.3)
  → Dense(64, relu)
  → Dense(1)  [regression]
```

In [ ]:
# Build per-game feature sequences
# These are the raw per-game stats (not rolling averages) — the LSTM will learn the patterns itself

SEQ_LEN = 10   # number of past games to use as input

SEQ_FEATURE_COLS = [
    'pts', 'trb', 'ast', 'stl', 'blk', 'tov', 'mp', 'fga', 'fta', '3pa',
    'fg_pct', 'ft_pct', '3p_pct', 'is_home', 'days_rest', 'is_b2b'
]
SEQ_FEATURE_COLS = [c for c in SEQ_FEATURE_COLS if c in df_clean.columns]

print(f'Sequence features ({len(SEQ_FEATURE_COLS)}): {SEQ_FEATURE_COLS}')

# Scale sequence features separately
seq_scaler = StandardScaler()
seq_data = df_clean[SEQ_FEATURE_COLS].fillna(0).values.astype(np.float32)
seq_data_sc = seq_scaler.fit_transform(seq_data)
df_seq = df_clean[['player', 'game_date', TARGET_STAT]].copy()
df_seq['seq_idx'] = np.arange(len(df_seq))

def build_sequences(df_subset, seq_data_scaled, feature_df, seq_len=10):
    """Build (X_seq, y) arrays where X_seq[i] = last seq_len games for that player."""
    X_seqs, y_vals, dates = [], [], []
    for player, group in feature_df.groupby('player'):
        group = group.sort_values('game_date')
        idxs = group.index.tolist()
        for i in range(seq_len, len(idxs)):
            # Use the previous seq_len games (no lookahead)
            seq_idxs = idxs[i - seq_len: i]
            target_idx = idxs[i]
            X_seqs.append(seq_data_scaled[seq_idxs])
            y_vals.append(df_subset.loc[target_idx, TARGET_STAT])
            dates.append(df_subset.loc[target_idx, 'game_date'])
    return np.array(X_seqs, dtype=np.float32), np.array(y_vals, dtype=np.float32), np.array(dates)

print('Building sequences (this may take a minute)...')
X_seq, y_seq, dates_seq = build_sequences(df_clean.reset_index(drop=True),
                                           seq_data_sc,
                                           df_clean.reset_index(drop=True),
                                           seq_len=SEQ_LEN)
print(f'Sequences built: X_seq={X_seq.shape}  y_seq={y_seq.shape}')

In [ ]:
# Temporal split
cutoff_ts = (df_clean['game_date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)).to_datetime64()
train_mask = dates_seq <= cutoff_ts
test_mask  = dates_seq >  cutoff_ts

X_seq_train, y_seq_train = X_seq[train_mask], y_seq[train_mask]
X_seq_test,  y_seq_test  = X_seq[test_mask],  y_seq[test_mask]
print(f'LSTM train: {X_seq_train.shape}  test: {X_seq_test.shape}')

In [ ]:
def build_lstm(seq_len, n_features, lstm_units=128, dropout_rate=0.3):
    inp = keras.Input(shape=(seq_len, n_features), name='sequence')

    x = layers.LSTM(lstm_units, return_sequences=False, name='lstm')(inp)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='NBA_LSTM')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss='mae'
    )
    return model


lstm_model = build_lstm(SEQ_LEN, len(SEQ_FEATURE_COLS))
lstm_model.summary()

In [ ]:
lstm_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1)
]

lstm_history = lstm_model.fit(
    X_seq_train, y_seq_train,
    validation_split=0.15,
    epochs=100,
    batch_size=256,
    callbacks=lstm_callbacks,
    verbose=1
)

lstm_preds = lstm_model.predict(X_seq_test, verbose=0).flatten()
lstm_mae = mean_absolute_error(y_seq_test, lstm_preds)
print(f'\nLSTM MAE: {lstm_mae:.3f}')

In [ ]:
# Training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lstm_history.history['loss'],     label='Train loss', linewidth=2)
axes[0].plot(lstm_history.history['val_loss'], label='Val loss',   linewidth=2)
axes[0].set_title('LSTM Training Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE')
axes[0].legend()

axes[1].scatter(y_seq_test[:500], lstm_preds[:500], alpha=0.4, s=20, color='coral')
mn, mx = y_seq_test.min(), y_seq_test.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_title(f'LSTM Prediction vs Actual ({TARGET_STAT.upper()})', fontweight='bold')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / f'lstm_{TARGET_STAT}_results.png', bbox_inches='tight')
plt.show()

---
# Final Comparison: XGBoost vs MLP vs LSTM

In [ ]:
# Load XGBoost production MAE from results.json
prod_results_path = MODELS_DIR / 'results.json'
xgb_mae = None
if prod_results_path.exists():
    with open(prod_results_path) as f:
        prod_results = json.load(f)
    xgb_mae = prod_results.get(TARGET_STAT, {}).get('mae')

models_compared = [
    ('XGBoost (production)', xgb_mae, '#2ecc71'),
    ('MLP', mlp_mae, '#3498db'),
    ('LSTM', lstm_mae, '#e74c3c'),
]

print(f'\n=== {TARGET_STAT.upper()} MAE Comparison ===')
for name, mae, _ in models_compared:
    if mae is not None:
        print(f'  {name:<25} {mae:.3f}')

fig, ax = plt.subplots(figsize=(8, 5))
names  = [m[0] for m in models_compared if m[1] is not None]
maes   = [m[1] for m in models_compared if m[1] is not None]
colors = [m[2] for m in models_compared if m[1] is not None]

bars = ax.bar(names, maes, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title(f'{TARGET_STAT.upper()} Regression MAE: Model Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('MAE (lower = better)')
ax.set_ylim(0, max(maes) * 1.25)

plt.tight_layout()
plt.savefig(OUT_DIR / f'model_comparison_{TARGET_STAT}.png', bbox_inches='tight')
plt.show()

best = min((m for m in models_compared if m[1] is not None), key=lambda x: x[1])
print(f'\nBest model for {TARGET_STAT.upper()}: {best[0]} (MAE={best[1]:.3f})')

## Save TF Models to Drive

In [ ]:
SAVE_TF_MODELS = False   # set to True to save

if SAVE_TF_MODELS:
    tf_dir = BASE_DIR / 'models_tf'
    tf_dir.mkdir(exist_ok=True)

    mlp.save(tf_dir / f'mlp_{TARGET_STAT}.keras')
    lstm_model.save(tf_dir / f'lstm_{TARGET_STAT}.keras')

    # Save scalers
    with open(tf_dir / f'scaler_mlp_{TARGET_STAT}.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    with open(tf_dir / f'scaler_lstm_{TARGET_STAT}.pkl', 'wb') as f:
        pickle.dump(seq_scaler, f)

    print(f'TF models saved to {tf_dir}')
else:
    print('SAVE_TF_MODELS is False — set to True to save models.')

---
## Notes

**If XGBoost wins:** That's expected for tabular sports data. XGBoost was designed for this.
The neural net experiments are still valuable — they confirm XGBoost is the right choice.

**If LSTM beats XGBoost on MAE:** The sequence model is capturing player momentum that
rolling averages miss. Consider integrating a lightweight LSTM into the production pipeline
as a secondary signal.

**Tuning ideas:**
- Increase `SEQ_LEN` from 10 to 15 or 20
- Add opponent features to the LSTM sequence (opponent def rating per game)
- Try a Bidirectional LSTM or a 1D CNN instead
- Add an attention layer on top of the LSTM to weight recent games higher